# Task 2: Citation Mapping - BERT Training (LR Ablation)

**Model:** bert-base-uncased  
**Phase:** 1 — Learning Rate Ablation  
**Run:** task2-kaggle-bert-lr5e5-bs32-ep3-window1

In [ ]:
import transformers, datasets, accelerate
print(f"✅ transformers: {transformers.__version__}")
print(f"✅ datasets: {datasets.__version__}")
print(f"✅ accelerate: {accelerate.__version__}")

In [ ]:
import wandb
import os

try:
    from kaggle_secrets import UserSecretsClient
    secrets = UserSecretsClient()
    key = secrets.get_secret("WANDB_API_KEY")
    os.environ["WANDB_API_KEY"] = key
    wandb.login(key=key, relogin=True)
    print("✅ Wandb logged in")
except Exception as e:
    print(f"⚠️ Wandb login failed: {e}")
    os.environ["WANDB_MODE"] = "disabled"
    print("⚠️ Wandb disabled")

In [ ]:
import math

# ═══════════════════════════════════════════════════════════
# ⚙️ CONFIG — Phase 1: Learning Rate Ablation
# ═══════════════════════════════════════════════════════════
LEARNING_RATE               = 5e-5
CONTEXT_MODE                = "window_1"
NEG_RATIO                   = 3
NUM_EPOCHS                  = 3
PER_DEVICE_TRAIN_BATCH_SIZE = 8
PER_DEVICE_EVAL_BATCH_SIZE  = 8
GRADIENT_ACCUMULATION_STEPS = 4       # effective batch = 32
WARMUP_RATIO                = 0.1
WEIGHT_DECAY                = 0.01
SEED                        = 42
MODEL_NAME                  = "bert-base-uncased"
MAX_LENGTH                  = 512

EFFECTIVE_BATCH_SIZE = PER_DEVICE_TRAIN_BATCH_SIZE * GRADIENT_ACCUMULATION_STEPS
RUN_NAME = f"task2-kaggle-bert-lr5e5-bs{EFFECTIVE_BATCH_SIZE}-ep{NUM_EPOCHS}-{CONTEXT_MODE}"

print(f"✅ Config loaded | LR={LEARNING_RATE} | eff_batch={EFFECTIVE_BATCH_SIZE} | epochs={NUM_EPOCHS} | context={CONTEXT_MODE}")

In [ ]:
import os

DATA_ROOT = "/kaggle/input/datasets/tathiyennhi/task2-citation-mapping/task2"

train_path = os.path.join(DATA_ROOT, "train")
val_path = os.path.join(DATA_ROOT, "val")

train_count = len([f for f in os.listdir(train_path) if f.endswith('.label')])
val_count = len([f for f in os.listdir(val_path) if f.endswith('.label')])

print(f"✅ Train: {train_count:,} files")
print(f"✅ Val: {val_count:,} files")

In [ ]:
import json
import re
import random
from pathlib import Path
from datasets import Dataset

random.seed(SEED)


def get_context(text, citation_id, mode='full'):
    if mode == 'full':
        return text

    window = int(mode.split('_')[1])
    sentences = re.split(r'(?<=[.!?])\s+', text)

    target_idx = -1
    for i, sent in enumerate(sentences):
        if citation_id in sent:
            target_idx = i
            break

    if target_idx == -1:
        return text

    start = max(0, target_idx - window)
    end = min(len(sentences), target_idx + window + 1)
    return ' '.join(sentences[start:end])


def load_task2_data(data_dir, max_files=None, neg_ratio=3, context_mode='full'):
    data_path = Path(data_dir)
    label_files = sorted(data_path.glob('*.label'))

    if max_files:
        label_files = label_files[:max_files]

    total_files = len(label_files)
    print(f'📊 Loading {total_files:,} files | Mode: {context_mode}')

    examples = []
    skipped = 0
    stats = {'positive': 0, 'negative': 0}

    for file_idx, label_file in enumerate(label_files):
        if (file_idx + 1) % 1000 == 0:
            print(f'⏳ {file_idx+1:,}/{total_files:,} | Examples: {len(examples):,}')

        in_file = label_file.with_suffix('.in')

        try:
            with open(in_file) as f:
                in_data = json.load(f)
            with open(label_file) as f:
                label_data = json.load(f)
        except:
            skipped += 1
            continue

        text = in_data.get('text', '')
        if not text:
            skipped += 1
            continue

        candidates = in_data.get('citation_candidates', [])
        bib_entries = in_data.get('bib_entries', {})
        correct_citation = label_data.get('correct_citation', {})

        if not correct_citation or not candidates or not bib_entries:
            skipped += 1
            continue

        for citation_id, correct_paper_id in correct_citation.items():
            context = get_context(text, citation_id, mode=context_mode)

            if correct_paper_id in bib_entries:
                paper = bib_entries[correct_paper_id]
                paper_text = f"{paper.get('title', '')}. {paper.get('abstract', '')}"
                examples.append({'text_a': context, 'text_b': paper_text, 'label': 1})
                stats['positive'] += 1

            neg_candidates = [c for c in candidates if c != correct_paper_id and c in bib_entries]
            neg_sample = random.sample(neg_candidates, min(neg_ratio, len(neg_candidates)))

            for neg_paper_id in neg_sample:
                paper = bib_entries[neg_paper_id]
                paper_text = f"{paper.get('title', '')}. {paper.get('abstract', '')}"
                examples.append({'text_a': context, 'text_b': paper_text, 'label': 0})
                stats['negative'] += 1

    print(f'\n✅ {len(examples):,} examples | Pos: {stats["positive"]:,} | Neg: {stats["negative"]:,} | Skip: {skipped}')
    return examples


print('=' * 60)
print(f'CONTEXT MODE: {CONTEXT_MODE} | NEG_RATIO: {NEG_RATIO}')
print('=' * 60)

train_examples = load_task2_data(train_path, neg_ratio=NEG_RATIO, context_mode=CONTEXT_MODE)
val_examples = load_task2_data(val_path, neg_ratio=NEG_RATIO, context_mode=CONTEXT_MODE)

train_dataset = Dataset.from_list(train_examples)
val_dataset = Dataset.from_list(val_examples)

print(f'\n✅ Train: {len(train_dataset):,} | Val: {len(val_dataset):,}')

In [ ]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
print(f"✅ Tokenizer: {MODEL_NAME}")


def tokenize_function(examples):
    return tokenizer(
        examples["text_a"],
        examples["text_b"],
        max_length=MAX_LENGTH,
        truncation=True,
        padding="max_length",
    )


print("Tokenizing train...")
train_tokenized = train_dataset.map(tokenize_function, batched=True, remove_columns=["text_a", "text_b"])

print("Tokenizing val...")
val_tokenized = val_dataset.map(tokenize_function, batched=True, remove_columns=["text_a", "text_b"])

print(f"\n✅ Train: {len(train_tokenized):,} | Val: {len(val_tokenized):,}")

In [ ]:
from transformers import AutoModelForSequenceClassification

model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=2)
print(f"✅ Model: {MODEL_NAME} (num_labels=2)")

In [ ]:
import numpy as np
from sklearn.metrics import accuracy_score, precision_recall_fscore_support


def compute_metrics(pred):
    labels = pred.label_ids
    preds = np.argmax(pred.predictions, axis=1)
    accuracy = accuracy_score(labels, preds)
    precision, recall, f1, _ = precision_recall_fscore_support(labels, preds, average="binary", pos_label=1)
    return {"accuracy": accuracy, "precision": precision, "recall": recall, "f1": f1}


print("✅ Metrics defined")

In [ ]:
from transformers import TrainingArguments, Trainer

WANDB_PROJECT  = "task2-citation-mapping"
CHECKPOINT_DIR = f"/kaggle/working/checkpoints/{RUN_NAME}"
SAVE_DIR       = f"/kaggle/working/models/{RUN_NAME}"
CONFIG_DIR     = f"/kaggle/working/configs/{RUN_NAME}.json"

try:
    if wandb.run is None:
        wandb.init(
            project=WANDB_PROJECT,
            name=RUN_NAME,
            config={
                "model":                       MODEL_NAME,
                "learning_rate":               LEARNING_RATE,
                "num_train_epochs":            NUM_EPOCHS,
                "per_device_train_batch_size": PER_DEVICE_TRAIN_BATCH_SIZE,
                "gradient_accumulation_steps": GRADIENT_ACCUMULATION_STEPS,
                "effective_batch_size":        EFFECTIVE_BATCH_SIZE,
                "warmup_ratio":                WARMUP_RATIO,
                "weight_decay":                WEIGHT_DECAY,
                "context_mode":                CONTEXT_MODE,
                "neg_ratio":                   NEG_RATIO,
                "seed":                        SEED,
                "max_length":                  MAX_LENGTH,
            },
            resume="allow",
        )
    report_to = "wandb"
    print("✅ Wandb initialized")
except Exception:
    report_to = "none"
    print("⚠️ Wandb not available")


training_args = TrainingArguments(
    output_dir=CHECKPOINT_DIR,
    num_train_epochs=NUM_EPOCHS,
    per_device_train_batch_size=PER_DEVICE_TRAIN_BATCH_SIZE,
    per_device_eval_batch_size=PER_DEVICE_EVAL_BATCH_SIZE,
    gradient_accumulation_steps=GRADIENT_ACCUMULATION_STEPS,
    learning_rate=LEARNING_RATE,
    weight_decay=WEIGHT_DECAY,
    warmup_ratio=WARMUP_RATIO,
    eval_strategy="epoch",
    save_strategy="epoch",
    save_total_limit=3,
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    greater_is_better=True,
    logging_dir="/kaggle/working/logs",
    logging_steps=50,
    report_to=report_to,
    fp16=True,
    bf16=False,
    dataloader_num_workers=2,
    seed=SEED,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_tokenized,
    eval_dataset=val_tokenized,
    compute_metrics=compute_metrics,
)

print(
    f"\n✅ Trainer ready"
    f"\n   LR={training_args.learning_rate}"
    f"\n   epochs={training_args.num_train_epochs}"
    f"\n   per_device_batch={training_args.per_device_train_batch_size}"
    f"\n   grad_acc={training_args.gradient_accumulation_steps}"
    f"\n   effective_batch={EFFECTIVE_BATCH_SIZE}"
    f"\n   warmup_ratio={training_args.warmup_ratio}"
)

In [ ]:
print("=" * 60)
print(f"🚀 TRAINING | {RUN_NAME}")
print("=" * 60)

trainer.train()

print("\n✅ Training complete!")

In [ ]:
print("📊 VALIDATION RESULTS")
print("=" * 60)

eval_results = trainer.evaluate()

for key, value in eval_results.items():
    if isinstance(value, float):
        print(f"{key}: {value:.4f}")
    else:
        print(f"{key}: {value}")

print("=" * 60)
print(f"\n✅ Accuracy:  {eval_results.get('eval_accuracy', 0):.2%}")
print(f"✅ Precision: {eval_results.get('eval_precision', 0):.2%}")
print(f"✅ Recall:    {eval_results.get('eval_recall', 0):.2%}")
print(f"✅ F1:        {eval_results.get('eval_f1', 0):.2%}")

In [ ]:
import os
import json
import csv
import shutil

os.makedirs(SAVE_DIR, exist_ok=True)
os.makedirs(os.path.dirname(CONFIG_DIR), exist_ok=True)
os.makedirs("/kaggle/working/results", exist_ok=True)

trainer.save_model(SAVE_DIR)
tokenizer.save_pretrained(SAVE_DIR)
shutil.make_archive(SAVE_DIR, 'zip', SAVE_DIR)
print(f"✅ Model saved: {SAVE_DIR}")

# Save config
config_data = {
    "run_name":                    RUN_NAME,
    "model":                       MODEL_NAME,
    "learning_rate":               LEARNING_RATE,
    "num_train_epochs":            NUM_EPOCHS,
    "per_device_train_batch_size": PER_DEVICE_TRAIN_BATCH_SIZE,
    "gradient_accumulation_steps": GRADIENT_ACCUMULATION_STEPS,
    "effective_batch_size":        EFFECTIVE_BATCH_SIZE,
    "warmup_ratio":                WARMUP_RATIO,
    "weight_decay":                WEIGHT_DECAY,
    "context_mode":                CONTEXT_MODE,
    "neg_ratio":                   NEG_RATIO,
    "seed":                        SEED,
    "max_length":                  MAX_LENGTH,
    "eval_f1":                     eval_results.get("eval_f1", 0),
    "eval_loss":                   eval_results.get("eval_loss", 0),
    "eval_accuracy":               eval_results.get("eval_accuracy", 0),
    "eval_precision":              eval_results.get("eval_precision", 0),
    "eval_recall":                 eval_results.get("eval_recall", 0),
}
with open(CONFIG_DIR, "w") as f:
    json.dump(config_data, f, indent=2)
print(f"✅ Config saved: {CONFIG_DIR}")

# Append to ablation summary CSV
summary_path = "/kaggle/working/results/ablation_summary.csv"
write_header = not os.path.exists(summary_path)
with open(summary_path, "a", newline="") as f:
    writer = csv.DictWriter(f, fieldnames=config_data.keys())
    if write_header:
        writer.writeheader()
    writer.writerow(config_data)
print(f"✅ Appended to: {summary_path}")

try:
    wandb.finish()
except:
    pass

print(f"\n🎉 DONE | {RUN_NAME}")
print(f"   F1:        {eval_results.get('eval_f1', 0):.4f}")
print(f"   Eval loss: {eval_results.get('eval_loss', 0):.4f}")